# 3D Volume Reconstruction — Local Slice Prediction Baseline

Context-aware pipeline: **[z−1, z, z+1] → z+1** prediction using a lightweight 2D U-Net.

## Approach
- **Task**: Predict adjacent slice `z+1` from 3-slice local context around `z`.
- **Architecture**: Single 2D U-Net with 3 encoder/decoder scales (`48/96/192`).
- **Positional context**: Optional (currently disabled via `USE_POSITIONAL_ENCODING = False`).
- **Loss**: `2.0×MSE + 0.5×(1−SSIM) + 0.1×FFT + 0.02×LPIPS + 0.02×Grad`.
- **Inference**: Bidirectional refinement to reduce drift and accumulation artifacts.

## Architecture Details
- `DoubleConv` blocks (Conv→BN→ReLU ×2) in encoder/decoder
- `ResidualBlock` in bottleneck for stronger gradient flow
- Attention gates on skip connections
- Input channels: `9` without positional maps, `12` with positional maps
- No masking on input: clean slice context → clean adjacent slice

## Dataset
- LGG + MU-Glioma slice stacks
- Training samples: triplet context `(z−1, z, z+1)` with fixed target `z+1`

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import cv2
from PIL import Image
import math
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# Notebook deps (safe to re-run)
%pip install -q nibabel lpips

# nibabel — used to reorient 3-D volumes to RAS+ standard
try:
    import nibabel as nib
    from nibabel.orientations import aff2axcodes, axcodes2ornt, ornt_transform, inv_ornt_aff
    NII_AVAILABLE = True
    print("nibabel available — volume orientation normalisation enabled")
except ImportError:
    NII_AVAILABLE = False
    print("nibabel not found  — install with:  pip install nibabel")
    print("  Continuing with per-dataset 2-D orientation correction instead.")

# LPIPS perceptual loss
try:
    import lpips
    LPIPS_AVAILABLE = True
    print("lpips available — perceptual loss enabled")
except ImportError:
    LPIPS_AVAILABLE = False
    print("lpips not found — install with: pip install lpips")

import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print(f'Using device: {device}')


## 2. Configuration Note — Positional Encoding

In [ ]:
# Toggle positional/semantic context without rewriting the pipeline.
# Set False to temporarily hide positional encoding (requested).
USE_POSITIONAL_ENCODING = False

if USE_POSITIONAL_ENCODING:
    print("Positional conditioning: enabled (normalized slice-depth channels appended)")
else:
    print("Positional conditioning: hidden/disabled (image context only)")

## 3. Training Pairs — 3-Slice Context (z−1, z, z+1) → Target (z+1)


In [ ]:
import re

# ─────────────────────────────────────────────────────────────────────────────
#  Orientation-correction flags (2-D TIF level)
#
#  LGG and MU-Glioma export their 2-D slices with different orientation
#  conventions.  Without correction the model learns conflicting spatial
#  mappings from the two datasets, which manifests as progressive rotation /
#  drift in sequential generation.
#
#  These constants apply a fixed numpy transform in SliceReconstructionDataset
#  so that every slice — regardless of source — is presented in the same
#  canonical orientation (match LGG's axial view).
#
#  How to audit: run the ORIENTATION AUDIT cell below once and visually
#  confirm that a random LGG slice and a random MU-Glioma slice look the
#  same "way up".  Flip the constant if they don't.
# ─────────────────────────────────────────────────────────────────────────────
ORIENT_LGG_FLIPUD       = False   # LGG is already in axial standard
ORIENT_MUGLIOMA_FLIPUD  = True    # MU-Glioma slices are stored with inverted z
ORIENT_MUGLIOMA_FLIPLR  = False   # set True if left-right also differs


def _natural_sort_key(path):
    """Sort key that orders slice_2 before slice_10 (numeric, not lexicographic)."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', path.name)]

def _collect_slices_lgg(patient_folder):
    """LGG: slices sit directly in the patient folder."""
    slices = [f for f in patient_folder.glob('*.tif') if '_mask' not in f.name]
    return sorted(slices, key=_natural_sort_key)

def _collect_slices_muglioma(patient_folder):
    """MU-Glioma: slices may be nested inside modality sub-folders.
    We take the modality folder that has the most slices.
    """
    slices = [f for f in patient_folder.rglob('*.tif') if '_mask' not in f.name.lower()]
    if not slices:
        slices = [f for f in patient_folder.rglob('*.png') if '_mask' not in f.name.lower()]
    return sorted(slices, key=_natural_sort_key)

# ── Data paths — Kaggle runtime with local fallback ────────────────────────
_KAGGLE_LGG      = Path('/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m')
_LOCAL_LGG       = Path('archive/kaggle_3m')
LGG_DATA_DIR     = _KAGGLE_LGG if _KAGGLE_LGG.exists() else _LOCAL_LGG

_KAGGLE_MU       = Path('/kaggle/input/datasets/prishapgpg/pkg-mu/PKG - MU-Glioma-Post/MU-Glioma-Post')
MU_GLIOMA_DATA_DIR = _KAGGLE_MU if _KAGGLE_MU.exists() else None  # no local fallback yet

# ── DATASET 1: LGG ────────────────────────────────────────────────────────
print("=" * 70)
print("DATASET 1: LGG (Lower Grade Glioma)")
print("=" * 70)
print(f"Directory : {LGG_DATA_DIR}  (exists: {LGG_DATA_DIR.exists()})")

lgg_patient_slices = {}
if LGG_DATA_DIR.exists():
    lgg_patient_folders = sorted(
        [f for f in LGG_DATA_DIR.iterdir() if f.is_dir() and f.name.startswith('TCGA')]
    )
    print(f"Patient folders found: {len(lgg_patient_folders)}")
    for pf in tqdm(lgg_patient_folders, desc="Scanning LGG"):
        slices = _collect_slices_lgg(pf)
        if slices:
            lgg_patient_slices[pf.name] = slices
else:
    print("⚠  LGG directory not found.")

lgg_slice_counts = [len(v) for v in lgg_patient_slices.values()]
if lgg_slice_counts:
    print(f"Patients with slices : {len(lgg_patient_slices)}")
    print(f"Slices/patient       : min={min(lgg_slice_counts)}  max={max(lgg_slice_counts)}  mean={np.mean(lgg_slice_counts):.1f}")
    print(f"Total LGG slices     : {sum(lgg_slice_counts):,}")

# ── DATASET 2: MU-Glioma ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("DATASET 2: MU-Glioma")
print("=" * 70)

mu_glioma_patient_slices = {}
if MU_GLIOMA_DATA_DIR is not None and MU_GLIOMA_DATA_DIR.exists():
    print(f"Directory : {MU_GLIOMA_DATA_DIR}")
    mu_patient_folders = sorted([f for f in MU_GLIOMA_DATA_DIR.iterdir() if f.is_dir()])
    print(f"Patient folders found: {len(mu_patient_folders)}")
    for pf in tqdm(mu_patient_folders, desc="Scanning MU-Glioma"):
        slices = _collect_slices_muglioma(pf)
        if slices:
            mu_glioma_patient_slices[pf.name] = slices
    mu_counts = [len(v) for v in mu_glioma_patient_slices.values()]
    if mu_counts:
        print(f"Patients with slices : {len(mu_glioma_patient_slices)}")
        print(f"Slices/patient       : min={min(mu_counts)}  max={max(mu_counts)}  mean={np.mean(mu_counts):.1f}")
        print(f"Total MU-Glioma slices: {sum(mu_counts):,}")
    else:
        print("⚠  No .tif slices found — check path or file extension.")
else:
    print("⚠  MU-Glioma directory not found — continuing with LGG only.")

# ── Combined + source registry ────────────────────────────────────────────
print("\n" + "=" * 70)
print("COMBINED DATASETS")
print("=" * 70)

patient_slices = {**lgg_patient_slices, **mu_glioma_patient_slices}

# Track which dataset each patient belongs to (used for orientation correction)
patient_source = {}
for pid in lgg_patient_slices:
    patient_source[pid] = 'lgg'
for pid in mu_glioma_patient_slices:
    patient_source[pid] = 'muglioma'

all_slice_counts = [len(v) for v in patient_slices.values()]
print(f"Total patients       : {len(patient_slices)}  "
      f"(LGG: {len(lgg_patient_slices)}  MU-Glioma: {len(mu_glioma_patient_slices)})")
print(f"Total slices         : {sum(all_slice_counts):,}")
print(f"Slices/patient       : min={min(all_slice_counts)}  max={max(all_slice_counts)}  "
      f"mean={np.mean(all_slice_counts):.1f}")
print(f"\nOrientation flags:")
print(f"  LGG       → flipud={ORIENT_LGG_FLIPUD}   fliplr=False")
print(f"  MU-Glioma → flipud={ORIENT_MUGLIOMA_FLIPUD}   fliplr={ORIENT_MUGLIOMA_FLIPLR}")

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, counts, title, color in zip(
    axes,
    [lgg_slice_counts if lgg_slice_counts else [0], all_slice_counts],
    ['LGG — Slice Distribution', 'Combined — Slice Distribution'],
    ['steelblue', 'seagreen'],
):
    ax.hist(counts, bins=25, edgecolor='black', color=color, alpha=0.75)
    ax.set_xlabel('Slices per patient'); ax.set_ylabel('Patients')
    ax.set_title(title, fontweight='bold'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Training pairs — 3-slice context + position metadata
#
#  Image input: [slice(z-1), slice(z), slice(z+1)]          -> 9 channels
#  Pos input  : [pos(z-1),   pos(z),   pos(z+1)] maps       -> 3 channels
#  Final input: concat(image_context, position_maps)        -> 12 channels
#  Target     : slice(z+1)                                  -> 3 channels
#
#  For z=0 boundary, we use reflected previous index (z-1 -> z+1).
# ─────────────────────────────────────────────────────────────────────────────
MAX_DELTA = 1   # fixed by design for (z -> z+1) target
MIN_SLICE_STD = 0.015

def _is_informative(path, threshold=MIN_SLICE_STD):
    try:
        img = np.array(Image.open(path)).astype(np.float32) / 255.0
        return img.std() > threshold
    except Exception:
        return False

print(f"Filtering trivial slices  (min std > {MIN_SLICE_STD}) …")

all_paths = set()
for slices in patient_slices.values():
    all_paths.update(slices)

informative = {p: _is_informative(p) for p in tqdm(all_paths, desc="Checking slices")}

n_total = len(all_paths)
n_ok = sum(informative.values())
print(f"  Informative slices: {n_ok:,} / {n_total:,}  "
      f"({100*n_ok/n_total:.1f}%)  —  {n_total-n_ok:,} trivial slices removed")

training_pairs = []

for patient_id, slices in tqdm(patient_slices.items(), desc="Creating triplet-context pairs"):
    n_slices = len(slices)
    if n_slices < 2:
        continue

    src = patient_source.get(patient_id, 'lgg')

    # z goes up to n_slices-2 because target is z+1
    for z in range(0, n_slices - 1):
        prev_idx = z - 1 if z > 0 else min(1, n_slices - 1)  # reflected boundary at z=0
        curr_idx = z
        next_idx = z + 1
        tgt_idx = z + 1

        p_prev = slices[prev_idx]
        p_curr = slices[curr_idx]
        p_next = slices[next_idx]
        p_tgt = slices[tgt_idx]

        # Require all context/target slices to be informative
        if (not informative.get(p_prev, True) or
            not informative.get(p_curr, True) or
            not informative.get(p_next, True) or
            not informative.get(p_tgt, True)):
            continue

        training_pairs.append({
            'patient_id':   patient_id,
            'prev_slice':   p_prev,
            'input_slice':  p_curr,
            'next_slice':   p_next,
            'target_slice': p_tgt,
            'delta':        1,
            'source':       src,
            'prev_idx':     prev_idx,
            'curr_idx':     curr_idx,
            'next_idx':     next_idx,
            'n_slices':     n_slices,
        })

total = len(training_pairs)
source_counts = Counter(p['source'] for p in training_pairs)
print(f"\nTotal training pairs (triplet context) : {total:,}")
print(f"Pairs per source                       : {dict(source_counts)}")
print("Input channels per sample              : 12  ([z-1|z|z+1] + 3 positional maps)")


## 4. Custom Dataset Class

In [ ]:
def _orient_slice(img: np.ndarray, source: str) -> np.ndarray:
    if source == 'muglioma':
        if ORIENT_MUGLIOMA_FLIPUD:
            img = np.flipud(img).copy()
        if ORIENT_MUGLIOMA_FLIPLR:
            img = np.fliplr(img).copy()
    elif source == 'lgg' and ORIENT_LGG_FLIPUD:
        img = np.flipud(img).copy()
    return img


def _norm_pos(idx: int, n_slices: int) -> float:
    """Normalize slice index to [0, 1] for semantic depth context."""
    denom = max(1, n_slices - 1)
    return float(np.clip(idx / denom, 0.0, 1.0))


class SliceReconstructionDataset(Dataset):
    """3-slice context reconstruction dataset.

    If USE_POSITIONAL_ENCODING is True:
      Input = 9 image channels + 3 positional channels = 12
    Else:
      Input = 9 image channels only
    Target = z+1 (3 channels)
    """

    def __init__(self, pairs, img_size=256, augment=False, patch_size=None):
        self.pairs = pairs
        self.img_size = img_size
        self.augment = augment
        self.patch_size = patch_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        source = pair.get('source', 'lgg')

        prev_img = np.array(Image.open(pair['prev_slice'])).astype(np.float32) / 255.0
        curr_img = np.array(Image.open(pair['input_slice'])).astype(np.float32) / 255.0
        next_img = np.array(Image.open(pair['next_slice'])).astype(np.float32) / 255.0
        target_img = np.array(Image.open(pair['target_slice'])).astype(np.float32) / 255.0

        prev_img = cv2.resize(prev_img, (self.img_size, self.img_size))
        curr_img = cv2.resize(curr_img, (self.img_size, self.img_size))
        next_img = cv2.resize(next_img, (self.img_size, self.img_size))
        target_img = cv2.resize(target_img, (self.img_size, self.img_size))

        prev_img = _orient_slice(prev_img, source)
        curr_img = _orient_slice(curr_img, source)
        next_img = _orient_slice(next_img, source)
        target_img = _orient_slice(target_img, source)

        if self.augment:
            if np.random.random() > 0.5:
                prev_img = np.flip(prev_img, axis=1).copy()
                curr_img = np.flip(curr_img, axis=1).copy()
                next_img = np.flip(next_img, axis=1).copy()
                target_img = np.flip(target_img, axis=1).copy()

            if np.random.random() > 0.5:
                angle = np.random.uniform(-10, 10)
                M = cv2.getRotationMatrix2D((self.img_size / 2, self.img_size / 2), angle, 1.0)
                prev_img = cv2.warpAffine(prev_img, M, (self.img_size, self.img_size))
                curr_img = cv2.warpAffine(curr_img, M, (self.img_size, self.img_size))
                next_img = cv2.warpAffine(next_img, M, (self.img_size, self.img_size))
                target_img = cv2.warpAffine(target_img, M, (self.img_size, self.img_size))

            if np.random.random() > 0.5:
                alpha = np.random.uniform(0.9, 1.1)
                beta = np.random.uniform(-0.05, 0.05)
                prev_img = np.clip(alpha * prev_img + beta, 0, 1)
                curr_img = np.clip(alpha * curr_img + beta, 0, 1)
                next_img = np.clip(alpha * next_img + beta, 0, 1)
                target_img = np.clip(alpha * target_img + beta, 0, 1)

        if self.patch_size is not None:
            h, w = curr_img.shape[:2]
            top = np.random.randint(0, h - self.patch_size + 1)
            left = np.random.randint(0, w - self.patch_size + 1)
            prev_img = prev_img[top:top + self.patch_size, left:left + self.patch_size]
            curr_img = curr_img[top:top + self.patch_size, left:left + self.patch_size]
            next_img = next_img[top:top + self.patch_size, left:left + self.patch_size]
            target_img = target_img[top:top + self.patch_size, left:left + self.patch_size]

        input_ctx = np.concatenate([prev_img, curr_img, next_img], axis=2)  # (H,W,9)

        if USE_POSITIONAL_ENCODING:
            n_slices = int(pair.get('n_slices', 2))
            prev_pos = _norm_pos(int(pair.get('prev_idx', 0)), n_slices)
            curr_pos = _norm_pos(int(pair.get('curr_idx', 0)), n_slices)
            next_pos = _norm_pos(int(pair.get('next_idx', 1)), n_slices)

            h, w = input_ctx.shape[:2]
            pos_maps = np.stack([
                np.full((h, w), prev_pos, dtype=np.float32),
                np.full((h, w), curr_pos, dtype=np.float32),
                np.full((h, w), next_pos, dtype=np.float32),
            ], axis=2)  # (H,W,3)

            model_input = np.concatenate([input_ctx, pos_maps], axis=2)      # (H,W,12)
        else:
            model_input = input_ctx                                           # (H,W,9)

        input_tensor = torch.from_numpy(model_input).permute(2, 0, 1)
        target_tensor = torch.from_numpy(target_img).permute(2, 0, 1)

        return {
            'input_slice': input_tensor,
            'target_slice': target_tensor,
            'delta': torch.tensor(1, dtype=torch.long),
        }


def audit_orientation(n=3):
    lgg_pairs = [p for p in training_pairs if p['source'] == 'lgg'][:n]
    mu_pairs = [p for p in training_pairs if p['source'] == 'muglioma'][:n]
    if not lgg_pairs and not mu_pairs:
        print("No pairs available for audit — run data-loading cells first.")
        return

    rows = max(len(lgg_pairs), len(mu_pairs))
    fig, axes = plt.subplots(rows, 4, figsize=(16, 4 * rows))
    if rows == 1:
        axes = axes[np.newaxis, :]
    axes[0, 0].set_title("LGG prev (corrected)", fontweight='bold')
    axes[0, 1].set_title("LGG current (corrected)", fontweight='bold')
    axes[0, 2].set_title("MU prev (corrected)", fontweight='bold')
    axes[0, 3].set_title("MU current (corrected)", fontweight='bold')

    for row in range(rows):
        for col_offset, pairs_subset in [(0, lgg_pairs), (2, mu_pairs)]:
            if row < len(pairs_subset):
                p = pairs_subset[row]
                src = p['source']
                for c, key in enumerate(['prev_slice', 'input_slice']):
                    img = np.array(Image.open(p[key])).astype(np.float32) / 255.0
                    img = cv2.resize(img, (256, 256))
                    img = _orient_slice(img, src)
                    axes[row, col_offset + c].imshow(img[:, :, 1] if img.ndim == 3 else img, cmap='gray')
                    axes[row, col_offset + c].axis('off')
    plt.suptitle("Orientation Audit — confirm all slices are head-up", fontweight='bold')
    plt.tight_layout()
    plt.show()

audit_orientation()

## 5. Model Architecture Components

### 5.1 Residual Block

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block with batch normalization"""
    
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        out = self.relu(out)
        return out

### 5.2 Attention Module

In [ ]:
class AttentionBlock(nn.Module):
    """Attention mechanism for feature refinement"""
    
    def __init__(self, channels):
        super(AttentionBlock, self).__init__()
        self.conv = nn.Conv2d(channels, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        attention = self.sigmoid(self.conv(x))
        return x * attention


class AttentionGate(nn.Module):
    """U-Net++ style attention gate for skip connections
    
    Improves feature fusion by learning which features to pass through.
    This typically adds 1-2 dB PSNR gain.
    """
    
    def __init__(self, in_channels, gating_channels):
        super(AttentionGate, self).__init__()
        
        self.W_gate = nn.Sequential(
            nn.Conv2d(gating_channels, in_channels, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(in_channels)
        )
        
        self.W_x = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(in_channels)
        )
        
        self.psi = nn.Sequential(
            nn.Conv2d(in_channels, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x, gating):
        """Apply attention gate
        
        Args:
            x: Skip connection features
            gating: Decoder features (from lower resolution)
        """
        g1 = self.W_gate(gating)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


### 5.3 Simple 2D U-Net — Slice-to-Slice Predictor

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive (Conv2d → BN → ReLU) layers."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


def _bilinear_up(in_ch, out_ch):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
    )


class SimpleUNet(nn.Module):
    """Lightweight 2D U-Net for 3-slice context prediction + positional channels.

    Input : [z-1, z, z+1] image context (9ch) + position maps (3ch) = 12 channels
    Output: predicted slice (z+1) -> 3 channels
    """

    def __init__(self, in_channels=12, out_channels=3, features=(48, 96, 192)):
        super().__init__()

        self.enc1  = DoubleConv(in_channels, features[0])
        self.res1  = ResidualBlock(features[0])

        self.enc2  = DoubleConv(features[0], features[1])
        self.res2  = ResidualBlock(features[1])

        self.enc3  = DoubleConv(features[1], features[2])
        self.res3  = ResidualBlock(features[2])

        self.pool  = nn.AvgPool2d(2)

        self.bottleneck = nn.Sequential(
            DoubleConv(features[2], features[2] * 2),
            ResidualBlock(features[2] * 2),
        )

        self.ag3 = AttentionGate(in_channels=features[2], gating_channels=features[2])
        self.ag2 = AttentionGate(in_channels=features[1], gating_channels=features[1])
        self.ag1 = AttentionGate(in_channels=features[0], gating_channels=features[0])

        self.up3   = _bilinear_up(features[2] * 2, features[2])
        self.dec3  = DoubleConv(features[2] * 2, features[2])
        self.dres3 = ResidualBlock(features[2])

        self.up2   = _bilinear_up(features[2], features[1])
        self.dec2  = DoubleConv(features[1] * 2, features[1])
        self.dres2 = ResidualBlock(features[1])

        self.up1   = _bilinear_up(features[1], features[0])
        self.dec1  = DoubleConv(features[0] * 2, features[0])
        self.dres1 = ResidualBlock(features[0])

        self.head  = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        # Global residual from input is not shape-compatible with 3-channel output
        # when input carries context + positional channels.
        e1 = self.res1(self.enc1(x))
        e2 = self.res2(self.enc2(self.pool(e1)))
        e3 = self.res3(self.enc3(self.pool(e2)))

        b  = self.bottleneck(self.pool(e3))

        g3 = self.up3(b)
        d3 = self.dres3(self.dec3(torch.cat([g3, self.ag3(e3, g3)], dim=1)))

        g2 = self.up2(d3)
        d2 = self.dres2(self.dec2(torch.cat([g2, self.ag2(e2, g2)], dim=1)))

        g1 = self.up1(d2)
        d1 = self.dres1(self.dec1(torch.cat([g1, self.ag1(e1, g1)], dim=1)))

        return torch.clamp(self.head(d1), 0.0, 1.0)


# ── Sanity check ────────────────────────────────────────────────────────────
_m = SimpleUNet(in_channels=12, out_channels=3)
_x = torch.randn(2, 12, 96, 96)
_y = _m(_x)
print(f"SimpleUNet output shape : {_y.shape}")
print(f"Total parameters        : {sum(p.numel() for p in _m.parameters()):,}")
print("  (input: 12 channels = 9 image context + 3 positional maps, output: 3 channels)")

_ct2d_found = [
    name for name, mod in _m.named_modules()
    if isinstance(mod, nn.ConvTranspose2d)
]
if _ct2d_found:
    print(f"\n⚠  ConvTranspose2d still present in: {_ct2d_found}")
else:
    print("\n✓  No ConvTranspose2d found — checkerboard fix is complete.")


### 5.4 PatchGAN Discriminator (Spectral Norm + Label Smoothing)

A 70×70 PatchGAN discriminator trained alongside the generator after a
**12-epoch generator-only warm-up phase**.

#### Stabilisation choices

| Technique | Why |
|---|---|
| **Spectral Normalization** on every conv layer | Bounds the discriminator Lipschitz constant to ≤ 1; most effective single stabilizer for medical GAN training; replaces BatchNorm entirely |
| **Generator-only warm-up** (12 epochs) | Builds a strong reconstruction baseline before the disc sees any samples; prevents a random discriminator from dominating early gradients |
| **Discriminator LR = 1e-5** (gen LR = 5e-5, ratio 5:1) | Keeps the discriminator intentionally weaker so it can't outrun the generator |
| **Label smoothing** (real=0.9, fake=0.1) | Prevents the discriminator from becoming overconfident (saturating logits); keeps gradients flowing to the generator |
| **LSGAN** objectives (MSE, not BCE) | Eliminates vanishing gradients that plague vanilla GAN; stable across the full training range |
| **Early-stopping counter reset** at disc activation | Prevents the brief PSNR fluctuation at epoch 12 from triggering premature stopping |


In [ ]:
import torch.nn.functional as F
from torch.nn.utils import spectral_norm as SN


class PatchGANDiscriminator(nn.Module):
    """Wavelet-informed PatchGAN discriminator (DISGAN-style).

    Standard PatchGANs score patches in pixel space only, which allows the
    discriminator to latch onto high-frequency grid artifacts (checkerboard)
    and amplify them by rewarding the generator for producing sharp-looking
    artifacts instead of anatomically correct textures.

    This variant adds a parallel *frequency path* that applies a single-level
    Haar DWT (pure-PyTorch, no external library) to the 6-channel input and
    processes the four subbands (LL, LH, HL, HH) with a separate SN-conv
    stream.  The two paths (spatial + frequency) are fused before the final
    scoring layers, forcing the discriminator to jointly evaluate:
      - Spatial coherence   (standard PatchGAN behavior)
      - Frequency plausibility  (DWT stream — penalises checkerboard grids
                                 which appear as energy spikes in HH/HL/LH
                                 subbands but not in natural MRI textures)

    Architecture
    ------------
    Input           : (B, 6, H, W)  [inp ∥ real_or_fake]
    Spatial path    : 2 × SN-Conv(stride=2) → (B, 2·ndf, H/4, W/4)
    Frequency path  : Haar-DWT → 4·6=24ch at H/2 × W/2
                      2 × SN-Conv(stride=1) → (B, 2·ndf, H/2, W/2)
                      AvgPool2d(2)          → (B, 2·ndf, H/4, W/4)
    Fusion          : cat → (B, 4·ndf, H/4, W/4)
                      SN-Conv(stride=2)     → (B, 4·ndf, H/8, W/8)
                      SN-Conv(stride=1)     → (B, 8·ndf, H/8, W/8)
                      SN-Conv(stride=1)     → (B, 1,    ~H/8, ~W/8)  patch map

    All convolutional layers use Spectral Normalization (no BatchNorm).
    Raw logits output for LSGAN / MSE objectives.
    """

    def __init__(self, in_channels: int = 6, ndf: int = 64):
        super().__init__()

        # ── Helpers ──────────────────────────────────────────────────────────
        def _sn(ic, oc, k=4, s=2, p=1):
            return nn.Sequential(
                SN(nn.Conv2d(ic, oc, kernel_size=k, stride=s, padding=p, bias=True)),
                nn.LeakyReLU(0.2, inplace=True),
            )

        # ── Spatial path (pixel domain, stride-2 downsampling) ────────────
        self.sp1 = _sn(in_channels, ndf,     k=4, s=2, p=1)   # H/2
        self.sp2 = _sn(ndf,         ndf * 2, k=4, s=2, p=1)   # H/4

        # ── Frequency path (wavelet domain, stride-1 to preserve H/2 res) ─
        # Haar DWT on the 6-ch input → 24 channels at H/2, W/2
        self.fr1 = _sn(in_channels * 4, ndf,     k=3, s=1, p=1)  # H/2 (stride-1)
        self.fr2 = _sn(ndf,              ndf * 2, k=3, s=1, p=1)  # H/2 (stride-1)

        # ── Fusion layers (operate on spatial H/4) ────────────────────────
        self.fu1 = _sn(ndf * 4, ndf * 4, k=4, s=2, p=1)   # H/8
        self.fu2 = _sn(ndf * 4, ndf * 8, k=4, s=1, p=1)   # H/8 (stride-1)
        self.out = SN(nn.Conv2d(ndf * 8, 1, kernel_size=4, stride=1, padding=1))

    # ── Pure-PyTorch single-level Haar DWT ───────────────────────────────────
    @staticmethod
    def _haar_dwt(x: torch.Tensor) -> torch.Tensor:
        """Single-level 2D Haar DWT — no external library required.

        Returns the four subbands (LL, LH, HL, HH) stacked along the
        channel dimension:  (B, C, H, W) → (B, 4·C, H//2, W//2).

        Haar filters:
          LL = ( x00 + x01 + x10 + x11 ) / 4   low-low  (smooth)
          LH = (-x00 - x01 + x10 + x11 ) / 4   low-high (horizontal edges)
          HL = (-x00 + x01 - x10 + x11 ) / 4   high-low (vertical edges)
          HH = ( x00 - x01 - x10 + x11 ) / 4   high-high (diagonal / grid)
        """
        H, W = x.shape[-2], x.shape[-1]
        x = x[:, :, :H // 2 * 2, :W // 2 * 2]     # ensure even dims
        x00 = x[:, :, 0::2, 0::2]
        x01 = x[:, :, 0::2, 1::2]
        x10 = x[:, :, 1::2, 0::2]
        x11 = x[:, :, 1::2, 1::2]
        LL = (x00 + x01 + x10 + x11) * 0.25
        LH = (-x00 - x01 + x10 + x11) * 0.25
        HL = (-x00 + x01 - x10 + x11) * 0.25
        HH = (x00 - x01 - x10 + x11) * 0.25
        return torch.cat([LL, LH, HL, HH], dim=1)  # (B, 4C, H/2, W/2)

    def forward(self, inp: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        inp    : (B, 3, H, W)  — conditioning input slice
        target : (B, 3, H, W)  — real or predicted adjacent slice

        Returns
        -------
        (B, 1, H', W') patch-level score map (raw logits for LSGAN)
        """
        x = torch.cat([inp, target], dim=1)   # (B, 6, H, W)

        # Spatial path
        sp = self.sp2(self.sp1(x))             # (B, 2·ndf, H/4, W/4)

        # Frequency path
        x_dwt = self._haar_dwt(x)             # (B, 24,    H/2, W/2)
        fr = self.fr2(self.fr1(x_dwt))        # (B, 2·ndf, H/2, W/2)
        fr = F.avg_pool2d(fr, 2)              # (B, 2·ndf, H/4, W/4)

        # Fuse & score
        fused = torch.cat([sp, fr], dim=1)    # (B, 4·ndf, H/4, W/4)
        return self.out(self.fu2(self.fu1(fused)))


# ── Sanity check ─────────────────────────────────────────────────────────────
_disc  = PatchGANDiscriminator(in_channels=6, ndf=64)
_inp   = torch.randn(2, 3, 256, 256)
_tgt   = torch.randn(2, 3, 256, 256)
_score = _disc(_inp, _tgt)
print(f"WaveletPatchGAN score map  : {_score.shape}")
print(f"Total discriminator params : {sum(p.numel() for p in _disc.parameters()):,}")
print("(Spatial path + Haar-DWT frequency path fused before scoring)")
print("(Spectral Norm on all layers, no BatchNorm, LSGAN-compatible logits)")
print()
# Verify DWT produces expected shape
_x6 = torch.randn(2, 6, 256, 256)
_dwt = PatchGANDiscriminator._haar_dwt(_x6)
print(f"Haar DWT (6ch 256×256) → {_dwt.shape}  [expected (2, 24, 128, 128)]")


## 6. Loss Function — MSE + SSIM

In [ ]:
_expected_in = 12 if USE_POSITIONAL_ENCODING else 9
print(f"SimpleUNet architecture: 3 scales, features=(48, 96, 192), input={_expected_in} channels.")
print("Loss: 2.0*MSE + 0.5*(1-SSIM) + 0.1*FFT_L1 + 0.02*LPIPS + 0.02*GradLoss")


## 7. Reconstruction Loss — `CombinedLoss`

$$L_{\text{rec}} = 2.0\cdot\text{MSE} + 0.5\cdot(1-\text{SSIM}) + 0.1\cdot\text{FFT\_L1} + 0.02\cdot\text{LPIPS} + 0.02\cdot\text{GradLoss}$$

| Term | Weight | Purpose |
|---|---|---|
| MSE | 2.00 | Pixel-level fidelity (heavily emphasized) |
| 1 − SSIM | 0.50 | Structural consistency |
| FFT L1 | 0.10 | Frequency-domain detail preservation |
| LPIPS (VGG) | 0.02 | Perceptual realism (light) |
| Sobel Gradient L1 | 0.02 | Edge/contour sharpness (light) |

Adversarial loss is gated by Phase-1 quality: it is enabled in Phase 2 only if
best Phase-1 validation PSNR exceeds **18 dB**; otherwise generator training keeps
`lambda_adv = 0.0` (reconstruction-only objective).


In [ ]:
class SobelGradientLoss(nn.Module):
    """L1 loss on Sobel gradient magnitude maps (edge/structure consistency)."""

    def __init__(self):
        super().__init__()
        kx = torch.tensor([[1., 0., -1.], [2., 0., -2.], [1., 0., -1.]], dtype=torch.float32)
        ky = torch.tensor([[1., 2., 1.], [0., 0., 0.], [-1., -2., -1.]], dtype=torch.float32)
        self.register_buffer("kx", kx.view(1, 1, 3, 3))
        self.register_buffer("ky", ky.view(1, 1, 3, 3))

    def _sobel_mag(self, x: torch.Tensor) -> torch.Tensor:
        c = x.shape[1]
        kx = self.kx.repeat(c, 1, 1, 1)
        ky = self.ky.repeat(c, 1, 1, 1)
        gx = F.conv2d(x, kx, padding=1, groups=c)
        gy = F.conv2d(x, ky, padding=1, groups=c)
        return torch.sqrt(gx * gx + gy * gy + 1e-8)

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return F.l1_loss(self._sobel_mag(pred), self._sobel_mag(target))


class CombinedLoss(nn.Module):
    """2.0*MSE + 0.5*(1-SSIM) + 0.1*FFT_L1 + 0.02*LPIPS + 0.02*GradLoss"""

    def __init__(self,
                 w_mse: float = 2.0,
                 w_ssim: float = 0.5,
                 w_fft: float = 0.1,
                 w_lpips: float = 0.02,
                 w_grad: float = 0.02,
                 device: str = 'cpu'):
        super().__init__()
        self.w_mse = w_mse
        self.w_ssim = w_ssim
        self.w_fft = w_fft
        self.w_lpips = w_lpips
        self.w_grad = w_grad

        self.mse = nn.MSELoss()
        self.grad_loss = SobelGradientLoss()

        if LPIPS_AVAILABLE:
            self.lpips = lpips.LPIPS(net='vgg').to(device)
            self.lpips.eval()
            for p in self.lpips.parameters():
                p.requires_grad = False
        else:
            self.lpips = None

    def _ssim_loss(self, pred: torch.Tensor, target: torch.Tensor) -> float:
        p = pred.detach().cpu().float().numpy()
        t = target.detach().cpu().float().numpy()
        vals = []
        for i in range(p.shape[0]):
            vals.append(ssim(
                t[i].transpose(1, 2, 0),
                p[i].transpose(1, 2, 0),
                data_range=1.0, channel_axis=2
            ))
        return 1.0 - float(np.mean(vals))

    @staticmethod
    def _fft_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred_f = pred.float()
        target_f = target.float()
        pred_mag = torch.abs(torch.fft.rfft2(pred_f, norm='ortho'))
        target_mag = torch.abs(torch.fft.rfft2(target_f, norm='ortho'))
        return F.l1_loss(pred_mag, target_mag)

    def _lpips_loss(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        if self.lpips is None:
            return torch.zeros((), device=pred.device)
        pred_11 = pred * 2.0 - 1.0
        target_11 = target * 2.0 - 1.0
        return self.lpips(pred_11, target_11).mean()

    def forward(self, pred: torch.Tensor, target: torch.Tensor):
        mse_l = self.mse(pred, target)
        ssim_l = self._ssim_loss(pred, target)
        fft_l = self._fft_loss(pred, target)
        lpips_l = self._lpips_loss(pred, target)
        grad_l = self.grad_loss(pred, target)

        total = (
            self.w_mse * mse_l +
            self.w_ssim * ssim_l +
            self.w_fft * fft_l +
            self.w_lpips * lpips_l +
            self.w_grad * grad_l
        )

        return (
            total,
            mse_l.item(),
            ssim_l,
            fft_l.item(),
            lpips_l.item(),
            grad_l.item(),
        )


SimpleLoss = CombinedLoss

print("CombinedLoss: 2.0*MSE + 0.5*(1-SSIM) + 0.1*FFT_L1 + 0.02*LPIPS + 0.02*GradLoss")
print("Adversarial term is phase-gated (enabled only if Phase-1 best PSNR > 18 dB).")


## 8. Metrics: PSNR and SSIM

In [ ]:
print("Loss: CombinedLoss  →  2.0*MSE + 0.5*(1-SSIM) + 0.1*FFT_L1 + 0.02*LPIPS + 0.02*GradLoss")
print("Adversarial term     →  enabled in Phase 2 only if Phase-1 PSNR > 18 dB")


### 8.1 PSNR / SSIM helper functions

In [ ]:
def calculate_psnr(pred, target):
    """Calculate PSNR between predicted and target images"""
    pred_np = pred.detach().cpu().numpy()
    target_np = target.detach().cpu().numpy()
    
    psnr_values = []
    for i in range(pred_np.shape[0]):
        # Transpose to (H, W, C)
        pred_img = pred_np[i].transpose(1, 2, 0)
        target_img = target_np[i].transpose(1, 2, 0)
        
        # Calculate PSNR
        psnr_val = psnr(target_img, pred_img, data_range=1.0)
        psnr_values.append(psnr_val)
    
    return np.mean(psnr_values)


def calculate_ssim(pred, target):
    """Calculate SSIM between predicted and target images"""
    pred_np = pred.detach().cpu().numpy()
    target_np = target.detach().cpu().numpy()
    
    ssim_values = []
    for i in range(pred_np.shape[0]):
        # Transpose to (H, W, C)
        pred_img = pred_np[i].transpose(1, 2, 0)
        target_img = target_np[i].transpose(1, 2, 0)
        
        # Calculate SSIM
        ssim_val = ssim(target_img, pred_img, data_range=1.0, channel_axis=2)
        ssim_values.append(ssim_val)
    
    return np.mean(ssim_values)

## 10. Data Splitting and DataLoader Setup

In [ ]:
# ── Patient-level train/val/test split (no data leakage) ──────────────────
patient_ids = list(patient_slices.keys())
train_patients, temp_patients = train_test_split(patient_ids, test_size=0.3, random_state=42)
val_patients, test_patients   = train_test_split(temp_patients, test_size=0.5, random_state=42)

print(f"Train patients : {len(train_patients)}")
print(f"Val patients   : {len(val_patients)}")
print(f"Test patients  : {len(test_patients)}")

# Filter pairs by split
train_set = set(train_patients)
val_set   = set(val_patients)
test_set  = set(test_patients)

train_pairs = [p for p in training_pairs if p['patient_id'] in train_set]
val_pairs   = [p for p in training_pairs if p['patient_id'] in val_set]
test_pairs  = [p for p in training_pairs if p['patient_id'] in test_set]

print(f"\nTrain pairs : {len(train_pairs):,}")
print(f"Val pairs   : {len(val_pairs):,}")
print(f"Test pairs  : {len(test_pairs):,}")

# ── Hyperparameters ────────────────────────────────────────────────────────
IMG_SIZE    = 256
PATCH_SIZE  = 96
BATCH_SIZE  = 4
GRADIENT_ACCUMULATION_STEPS = 4   # effective batch = 4 × 4 = 16

print(f"\nPatch size     : {PATCH_SIZE}×{PATCH_SIZE}")
print(f"Batch size     : {BATCH_SIZE}  (effective {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS} with grad-accum)")

# ── Datasets ───────────────────────────────────────────────────────────────
train_dataset = SliceReconstructionDataset(
    train_pairs, img_size=IMG_SIZE, augment=True,  patch_size=PATCH_SIZE)
val_dataset   = SliceReconstructionDataset(
    val_pairs,   img_size=IMG_SIZE, augment=False, patch_size=None)
test_dataset  = SliceReconstructionDataset(
    test_pairs,  img_size=IMG_SIZE, augment=False, patch_size=None)

# ── DataLoaders ────────────────────────────────────────────────────────────
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

expected_in = 12 if USE_POSITIONAL_ENCODING else 9
print(f"Positional encoding enabled: {USE_POSITIONAL_ENCODING}")

# ── Batch-shape sanity check ───────────────────────────────────────────────
try:
    _dbg_batch = next(iter(train_loader))
    _x = _dbg_batch['input_slice']
    _y = _dbg_batch['target_slice']
    print("\nBatch shape sanity check:")
    print(f"  input_slice  : {_x.shape}  (expected [B, {expected_in}, H, W])")
    print(f"  target_slice : {_y.shape}  (expected [B, 3, H, W])")
except StopIteration:
    print("\nBatch shape sanity check skipped: train_loader is empty.")


## 11. Model Initialization

In [ ]:
# ── Training Hyperparameters ───────────────────────────────────────────────
IMG_SIZE    = 256
PATCH_SIZE  = 96
BATCH_SIZE  = 4
GRADIENT_ACCUMULATION_STEPS = 4      # effective batch = 16

WARMUP_EPOCHS           = 3
START_LR                = 1e-5
PEAK_LR                 = 1e-5
DISC_LR                 = 1e-5
EARLY_STOPPING_PATIENCE = 20
LAMBDA_ADV              = 0.005

INPUT_CHANNELS = 12 if USE_POSITIONAL_ENCODING else 9

# ── Generator ───────────────────────────────────────────────────────────────
model = SimpleUNet(in_channels=INPUT_CHANNELS, out_channels=3, features=(48, 96, 192)).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Architecture     : SimpleUNet  (3 scales, features=48/96/192)")
print(f"Positional enc.  : {USE_POSITIONAL_ENCODING}")
print(f"Input / Output   : {INPUT_CHANNELS}-channel input -> 3-channel target (z+1)")
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")

# ── Discriminator (conditioned on center image slice + target/pred) ─────────
# Input to D remains 6 channels: center context image slice (3ch) + real/fake target (3ch)
disc = PatchGANDiscriminator(in_channels=6, ndf=64).to(device)

disc_total = sum(p.numel() for p in disc.parameters())
print(f"\nDiscriminator    : PatchGANDiscriminator  (SN, 70×70, ndf=64)")
print(f"Disc params      : {disc_total:,}")

criterion = CombinedLoss(
    w_mse=2.0, w_ssim=0.5, w_fft=0.1, w_lpips=0.02, w_grad=0.02,
    device=str(device)
).to(device)
print("\nRec. Loss        : 2.0*MSE + 0.5*(1-SSIM) + 0.1*FFT_L1 + 0.02*LPIPS + 0.02*GradLoss")
print(f"Adv. weight(base): λ_adv = {LAMBDA_ADV}  (phase-gated by Phase-1 PSNR > 18 dB)")

optimizer = torch.optim.Adam(model.parameters(), lr=START_LR,
                             betas=(0.5, 0.999), weight_decay=1e-5)

disc_optimizer = torch.optim.Adam(disc.parameters(), lr=DISC_LR,
                                  betas=(0.5, 0.999))

scheduler = None

scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

print(f"\nPatch size       : {PATCH_SIZE}×{PATCH_SIZE}")
print(f"Batch / Eff.     : {BATCH_SIZE} / {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Early stopping   : {EARLY_STOPPING_PATIENCE} (used in both phases)")
print(f"Gen / Disc LR    : {START_LR:.0e} / {DISC_LR:.0e}")
print(f"λ_adv(base)      : {LAMBDA_ADV}")
print(f"AMP              : {'enabled' if scaler else 'disabled'}")


## 12. Training Functions

In [ ]:
REAL_LABEL = 0.9
FAKE_LABEL = 0.1


def _center_from_context(inp_ctx: torch.Tensor) -> torch.Tensor:
    """Extract center image slice (z) from input layout.

    Input layout: [z-1(3ch), z(3ch), z+1(3ch), pos(z-1), pos(z), pos(z+1)]
    """
    return inp_ctx[:, 3:6, :, :]


def _zscore_slice(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Per-sample z-score for a 3-channel slice tensor [B,3,H,W]."""
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    std = x.std(dim=(1, 2, 3), keepdim=True, unbiased=False)
    return (x - mean) / (std + eps)


def normalize_context_input(inp_ctx: torch.Tensor) -> torch.Tensor:
    """Normalize context image slices; keep positional channels unchanged.

    Input channels:
      - 9  : [z-1(3), z(3), z+1(3)]
      - 12 : [z-1(3), z(3), z+1(3), pos(z-1), pos(z), pos(z+1)]

    Returns normalized context with same shape.
    """
    prev = _zscore_slice(inp_ctx[:, 0:3, :, :])
    curr = _zscore_slice(inp_ctx[:, 3:6, :, :])
    nxt  = _zscore_slice(inp_ctx[:, 6:9, :, :])
    norm_ctx = torch.cat([prev, curr, nxt], dim=1)  # [B,9,H,W] ~ N(0,1)

    if inp_ctx.shape[1] > 9:
        return torch.cat([norm_ctx, inp_ctx[:, 9:, :, :]], dim=1)
    return norm_ctx


def train_epoch(model, disc, dataloader, criterion, optimizer, disc_optimizer,
                device, scaler=None, grad_accum=1, ss_prob=0.0,
                lambda_adv=0.005, disc_active=True):
    model.train()
    if disc_active:
        disc.train()
    else:
        disc.eval()

    running = {
        'loss': 0.0, 'mse': 0.0, 'ssim_loss': 0.0, 'fft_loss': 0.0,
        'lpips_loss': 0.0, 'grad_loss': 0.0,
        'psnr': 0.0, 'ssim': 0.0, 'disc_loss': 0.0, 'adv_loss': 0.0
    }
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(dataloader):
        inp_raw = batch['input_slice'].to(device)      # raw context in [0,1] + optional pos
        target = batch['target_slice'].to(device)      # target stays [0,1] (for MSE)

        if ss_prob > 0.0:
            with torch.no_grad():
                inp_pred = model(normalize_context_input(inp_raw))
            mask = (torch.rand(inp_raw.shape[0], 1, 1, 1, device=device) < ss_prob).float()
            center_raw = _center_from_context(inp_raw)
            center_raw = (mask * inp_pred + (1.0 - mask) * center_raw).detach()
            inp_raw = torch.cat([inp_raw[:, 0:3], center_raw, inp_raw[:, 6:]], dim=1)

        inp = normalize_context_input(inp_raw)
        cond = _center_from_context(inp_raw)  # discriminator condition in [0,1]

        disc_loss_val = 0.0
        if disc_active:
            with torch.no_grad():
                fake = model(inp)

            disc_optimizer.zero_grad()
            real_score = disc(cond, target)
            fake_score = disc(cond, fake.detach())

            loss_real = F.mse_loss(real_score, torch.full_like(real_score, REAL_LABEL))
            loss_fake = F.mse_loss(fake_score, torch.full_like(fake_score, FAKE_LABEL))
            disc_loss = 0.5 * (loss_real + loss_fake)

            disc_loss.backward()
            torch.nn.utils.clip_grad_norm_(disc.parameters(), 1.0)
            disc_optimizer.step()
            disc_loss_val = disc_loss.item()

        adv_loss_val = 0.0
        if scaler is not None:
            with torch.cuda.amp.autocast():
                out = model(inp)
                total_rec, mse_l, ssim_l, fft_l, lpips_l, grad_l = criterion(out, target)
                if disc_active:
                    adv_score = disc(cond, out)
                    adv_loss = F.mse_loss(adv_score, torch.full_like(adv_score, REAL_LABEL))
                    gen_loss = total_rec + lambda_adv * adv_loss
                    adv_loss_val = adv_loss.item()
                else:
                    gen_loss = total_rec

            scaler.scale(gen_loss / grad_accum).backward()
            if (batch_idx + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
        else:
            out = model(inp)
            total_rec, mse_l, ssim_l, fft_l, lpips_l, grad_l = criterion(out, target)
            if disc_active:
                adv_score = disc(cond, out)
                adv_loss = F.mse_loss(adv_score, torch.full_like(adv_score, REAL_LABEL))
                gen_loss = total_rec + lambda_adv * adv_loss
                adv_loss_val = adv_loss.item()
            else:
                gen_loss = total_rec

            (gen_loss / grad_accum).backward()
            if (batch_idx + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()

        with torch.no_grad():
            psnr_val = calculate_psnr(out, target)
            ssim_val = calculate_ssim(out, target)

        running['loss'] += total_rec.item()
        running['mse'] += mse_l
        running['ssim_loss'] += ssim_l
        running['fft_loss'] += fft_l
        running['lpips_loss'] += lpips_l
        running['grad_loss'] += grad_l
        running['psnr'] += psnr_val
        running['ssim'] += ssim_val
        running['disc_loss'] += disc_loss_val
        running['adv_loss'] += adv_loss_val

        if device.type == 'cuda':
            torch.cuda.empty_cache()

    n = len(dataloader)
    return {k: v / n for k, v in running.items()}


def validate_epoch(model, dataloader, criterion, device, scaler=None):
    model.eval()
    running = {
        'loss': 0.0, 'mse': 0.0, 'ssim_loss': 0.0, 'fft_loss': 0.0,
        'lpips_loss': 0.0, 'grad_loss': 0.0,
        'psnr': 0.0, 'ssim': 0.0
    }

    with torch.no_grad():
        for batch in dataloader:
            inp_raw = batch['input_slice'].to(device)
            inp = normalize_context_input(inp_raw)
            target = batch['target_slice'].to(device)

            if scaler is not None:
                with torch.cuda.amp.autocast():
                    out = model(inp)
                    total_loss, mse_l, ssim_l, fft_l, lpips_l, grad_l = criterion(out, target)
            else:
                out = model(inp)
                total_loss, mse_l, ssim_l, fft_l, lpips_l, grad_l = criterion(out, target)

            psnr_val = calculate_psnr(out, target)
            ssim_val = calculate_ssim(out, target)

            running['loss'] += total_loss.item()
            running['mse'] += mse_l
            running['ssim_loss'] += ssim_l
            running['fft_loss'] += fft_l
            running['lpips_loss'] += lpips_l
            running['grad_loss'] += grad_l
            running['psnr'] += psnr_val
            running['ssim'] += ssim_val

            if device.type == 'cuda':
                torch.cuda.empty_cache()

    n = len(dataloader)
    return {k: v / n for k, v in running.items()}


## 13. Training Loop

## 13.1. Load Checkpoint (Optional - for resuming training)

In [ ]:
import os

_HISTORY_KEYS_TRAIN = (
    'loss', 'mse', 'ssim_loss', 'fft_loss', 'lpips_loss', 'grad_loss',
    'psnr', 'ssim', 'disc_loss', 'adv_loss'
)
_HISTORY_KEYS_VAL = (
    'loss', 'mse', 'ssim_loss', 'fft_loss', 'lpips_loss', 'grad_loss',
    'psnr', 'ssim'
)

def _empty_history():
    h = {f'train_{k}': [] for k in _HISTORY_KEYS_TRAIN}
    h.update({f'val_{k}': [] for k in _HISTORY_KEYS_VAL})
    return h

def load_checkpoint_compat(path, map_location):
    """Compat loader across PyTorch versions.

    - PyTorch >=2.6 defaults to weights_only=True, which can fail for full checkpoints.
    - Older versions may not accept the weights_only kwarg.
    """
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

checkpoint_files = [
    'checkpoint_epoch_12.pth',
    '3d_reconstruction_final.pth',
    'best_3d_reconstruction.pth',
]
CHECKPOINT_PATH = next((f for f in checkpoint_files if os.path.exists(f)), None)

RESUME_TRAINING = False
start_epoch = 0
best_val_psnr = 0.0
history = _empty_history()

if CHECKPOINT_PATH:
    print(f"Found checkpoint: {CHECKPOINT_PATH}")
    ckpt = load_checkpoint_compat(CHECKPOINT_PATH, map_location=device)

    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'])
        if 'disc_state_dict' in ckpt:
            disc.load_state_dict(ckpt['disc_state_dict'])
            print("  Discriminator weights restored.")
        if 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'disc_opt_state_dict' in ckpt:
            disc_optimizer.load_state_dict(ckpt['disc_opt_state_dict'])
        start_epoch = ckpt.get('epoch', 0)
        best_val_psnr = ckpt.get('best_val_psnr', 0.0)
        history = ckpt.get('history', _empty_history())
        RESUME_TRAINING = True
        print(f"✓ Resuming from epoch {start_epoch}  (best PSNR so far: {best_val_psnr:.2f} dB)")
        for _ in range(max(0, start_epoch - WARMUP_EPOCHS)):
            scheduler.step()
        print(f"  CosineAnnealingLR fast-forwarded {max(0, start_epoch - WARMUP_EPOCHS)} steps")
    else:
        model.load_state_dict(ckpt)
        print("✓ Generator weights loaded  (no full checkpoint — starting fresh counters)")
else:
    print("No checkpoint found — starting fresh training.")


## 13.2. Training Configuration

In [ ]:
PHASE1_EPOCHS           = 100
PHASE2_EPOCHS           = 100
PHASE1_PATIENCE         = 20
PHASE2_PATIENCE         = 20
CHECKPOINT_INTERVAL     = 5

SS_FACTOR_START         = 0.0
SS_FACTOR_END           = 0.1
SS_EPOCHS               = 200

LAMBDA_ADV              = 0.005
ADV_ENABLE_PSNR_THRESH  = 18.0
BEST_GEN_PATH           = 'best_generator.pth'

_TRAIN_KEYS = (
    'loss', 'mse', 'ssim_loss', 'fft_loss', 'lpips_loss', 'grad_loss',
    'psnr', 'ssim', 'disc_loss', 'adv_loss'
)
_VAL_KEYS = (
    'loss', 'mse', 'ssim_loss', 'fft_loss', 'lpips_loss', 'grad_loss',
    'psnr', 'ssim'
)

def _ss_prob(epoch_idx: int) -> float:
    """Linear scheduled-sampling ramp: start -> end over SS_EPOCHS."""
    if SS_EPOCHS <= 0:
        return float(SS_FACTOR_END)
    t = min(1.0, max(0.0, epoch_idx / float(SS_EPOCHS)))
    return float(SS_FACTOR_START + (SS_FACTOR_END - SS_FACTOR_START) * t)

if not RESUME_TRAINING:
    history = {f'train_{k}': [] for k in _TRAIN_KEYS}
    history.update({f'val_{k}': [] for k in _VAL_KEYS})

best_gen_val_psnr = 0.0
best_val_psnr = 0.0

# Rebuild scheduler for the full two-phase duration
TOTAL_EPOCHS = PHASE1_EPOCHS + PHASE2_EPOCHS
NUM_EPOCHS = TOTAL_EPOCHS  # keep downstream summary cells compatible
WARMUP_EPOCHS = min(WARMUP_EPOCHS, TOTAL_EPOCHS)
_sched_T_max = max(1, TOTAL_EPOCHS - WARMUP_EPOCHS)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=_sched_T_max, eta_min=START_LR)

print("=" * 72)
print("TWO-PHASE GAN TRAINING (UPDATED)")
print("=" * 72)
print(f"  Phase 1 (generator-only): max epochs={PHASE1_EPOCHS}, early stop patience={PHASE1_PATIENCE}")
print(f"  Phase 2 (GAN-active)    : max epochs={PHASE2_EPOCHS}, early stop patience={PHASE2_PATIENCE}")
print(f"  Scheduled sampling      : start={SS_FACTOR_START}, end={SS_FACTOR_END}, epochs={SS_EPOCHS}")
print(f"  Adv gate                : enable λ_adv only if Phase-1 best PSNR > {ADV_ENABLE_PSNR_THRESH:.1f} dB")
print(f"  Best Phase-1 generator checkpoint: {BEST_GEN_PATH}")
print(f"  Positional encoding enabled: {USE_POSITIONAL_ENCODING}")
print("=" * 72)

global_epoch = 0
phase1_stop_counter = 0
phase2_stop_counter = 0

# ── Phase 1: Generator-only training ───────────────────────────────────────
for local_epoch in range(PHASE1_EPOCHS):
    disc_active = False
    ss_prob = _ss_prob(global_epoch)
    global_epoch += 1

    print(f"\n[Phase 1] Epoch {local_epoch + 1}/{PHASE1_EPOCHS}  (global {global_epoch}/{TOTAL_EPOCHS}, ss={ss_prob:.3f})")
    print("-" * 72)

    train_metrics = train_epoch(
        model, disc, train_loader, criterion, optimizer, disc_optimizer,
        device, scaler=scaler, grad_accum=GRADIENT_ACCUMULATION_STEPS,
        ss_prob=ss_prob, lambda_adv=0.0, disc_active=disc_active)

    val_metrics = validate_epoch(model, val_loader, criterion, device, scaler=scaler)

    if global_epoch <= WARMUP_EPOCHS:
        lr_now = START_LR + (PEAK_LR - START_LR) * global_epoch / max(1, WARMUP_EPOCHS)
        for pg in optimizer.param_groups:
            pg['lr'] = lr_now
        print(f"  LR (warm-up): {lr_now:.2e}")
    else:
        scheduler.step()
        print(f"  LR (gen): {optimizer.param_groups[0]['lr']:.2e}")

    for k in _TRAIN_KEYS:
        history[f'train_{k}'].append(train_metrics[k])
    for k in _VAL_KEYS:
        history[f'val_{k}'].append(val_metrics[k])

    print(
        f"  Train Rec: {train_metrics['loss']:.4f} | "
        f"PSNR: {train_metrics['psnr']:.2f} dB | SSIM: {train_metrics['ssim']:.4f} | "
        f"LPIPS: {train_metrics['lpips_loss']:.4f}"
    )
    print(
        f"  Val   Rec: {val_metrics['loss']:.4f} | "
        f"PSNR: {val_metrics['psnr']:.2f} dB | SSIM: {val_metrics['ssim']:.4f} | "
        f"LPIPS: {val_metrics['lpips_loss']:.4f}"
    )

    if val_metrics['psnr'] > best_gen_val_psnr:
        best_gen_val_psnr = val_metrics['psnr']
        torch.save({
            'epoch':            global_epoch,
            'phase_epoch':      local_epoch + 1,
            'phase':            'generator_only',
            'model_state_dict': model.state_dict(),
            'best_val_psnr':    best_gen_val_psnr,
            'use_positional':   USE_POSITIONAL_ENCODING,
        }, BEST_GEN_PATH)
        print(f"  ✓ Best generator (Phase 1) saved: {BEST_GEN_PATH}  (PSNR: {best_gen_val_psnr:.2f} dB)")
        phase1_stop_counter = 0
    else:
        phase1_stop_counter += 1
        print(f"  [Phase 1 patience: {phase1_stop_counter}/{PHASE1_PATIENCE}] best={best_gen_val_psnr:.2f} dB")

    if (global_epoch % CHECKPOINT_INTERVAL) == 0:
        torch.save({
            'epoch':                  global_epoch,
            'phase':                  'phase1',
            'model_state_dict':       model.state_dict(),
            'disc_state_dict':        disc.state_dict(),
            'optimizer_state_dict':   optimizer.state_dict(),
            'disc_opt_state_dict':    disc_optimizer.state_dict(),
            'best_val_psnr':          best_val_psnr,
            'best_gen_val_psnr':      best_gen_val_psnr,
            'history':                history,
        }, f'checkpoint_epoch_{global_epoch}.pth')
        print(f"  ✓ Checkpoint saved: checkpoint_epoch_{global_epoch}.pth")

    if phase1_stop_counter >= PHASE1_PATIENCE:
        print(f"\nEarly stopping Phase 1 at epoch {local_epoch + 1}/{PHASE1_EPOCHS} (patience={PHASE1_PATIENCE}).")
        break

# ── Handoff: load best Phase-1 generator before Phase 2 ───────────────────
if os.path.exists(BEST_GEN_PATH):
    _gen_ckpt = load_checkpoint_compat(BEST_GEN_PATH, map_location=device)
    if isinstance(_gen_ckpt, dict) and 'model_state_dict' in _gen_ckpt:
        model.load_state_dict(_gen_ckpt['model_state_dict'])
        _loaded_psnr = _gen_ckpt.get('best_val_psnr', None)
    else:
        model.load_state_dict(_gen_ckpt)
        _loaded_psnr = None

    if _loaded_psnr is not None:
        print(f"\n✓ Loaded best Phase-1 generator from {BEST_GEN_PATH} (PSNR: {_loaded_psnr:.2f} dB)")
    else:
        print(f"\n✓ Loaded best Phase-1 generator from {BEST_GEN_PATH}")
else:
    print(f"\n⚠ Best generator checkpoint not found at {BEST_GEN_PATH}; Phase 2 will use current weights.")

# Adversarial gating based on Phase-1 quality
phase2_lambda_adv = LAMBDA_ADV if best_gen_val_psnr > ADV_ENABLE_PSNR_THRESH else 0.0

# reset GAN-phase best tracker and patience
best_val_psnr = 0.0
phase2_stop_counter = 0
print("\n★ Starting Phase 2 (GAN active)")
print(f"  Label smoothing: real={REAL_LABEL}, fake={FAKE_LABEL}")
print(f"  Phase-1 best PSNR: {best_gen_val_psnr:.2f} dB")
print(f"  λ_adv used in Phase 2: {phase2_lambda_adv} (threshold: {ADV_ENABLE_PSNR_THRESH:.1f} dB)")

# ── Phase 2: GAN training ───────────────────────────────────────────────────
for local_epoch in range(PHASE2_EPOCHS):
    disc_active = True
    ss_prob = _ss_prob(global_epoch)
    global_epoch += 1

    print(f"\n[Phase 2] Epoch {local_epoch + 1}/{PHASE2_EPOCHS}  (global {global_epoch}/{TOTAL_EPOCHS}, ss={ss_prob:.3f})")
    print("-" * 72)

    train_metrics = train_epoch(
        model, disc, train_loader, criterion, optimizer, disc_optimizer,
        device, scaler=scaler, grad_accum=GRADIENT_ACCUMULATION_STEPS,
        ss_prob=ss_prob, lambda_adv=phase2_lambda_adv, disc_active=disc_active)

    val_metrics = validate_epoch(model, val_loader, criterion, device, scaler=scaler)

    if global_epoch <= WARMUP_EPOCHS:
        lr_now = START_LR + (PEAK_LR - START_LR) * global_epoch / max(1, WARMUP_EPOCHS)
        for pg in optimizer.param_groups:
            pg['lr'] = lr_now
        print(f"  LR (warm-up): {lr_now:.2e}")
    else:
        scheduler.step()
        print(f"  LR (gen): {optimizer.param_groups[0]['lr']:.2e}  LR (disc): {DISC_LR:.0e}")

    for k in _TRAIN_KEYS:
        history[f'train_{k}'].append(train_metrics[k])
    for k in _VAL_KEYS:
        history[f'val_{k}'].append(val_metrics[k])

    print(
        f"  Train Rec: {train_metrics['loss']:.4f} | "
        f"PSNR: {train_metrics['psnr']:.2f} dB | SSIM: {train_metrics['ssim']:.4f} | "
        f"D: {train_metrics['disc_loss']:.4f} | Adv: {train_metrics['adv_loss']:.4f}"
    )
    print(
        f"  Val   Rec: {val_metrics['loss']:.4f} | "
        f"PSNR: {val_metrics['psnr']:.2f} dB | SSIM: {val_metrics['ssim']:.4f} | "
        f"LPIPS: {val_metrics['lpips_loss']:.4f}"
    )

    if val_metrics['psnr'] > best_val_psnr:
        best_val_psnr = val_metrics['psnr']
        torch.save({
            'epoch':            global_epoch,
            'phase_epoch':      local_epoch + 1,
            'phase':            'gan',
            'model_state_dict': model.state_dict(),
            'disc_state_dict':  disc.state_dict(),
            'best_val_psnr':    best_val_psnr,
            'use_positional':   USE_POSITIONAL_ENCODING,
            'phase2_lambda_adv': phase2_lambda_adv,
        }, 'best_3d_reconstruction.pth')
        print(f"  ✓ Best GAN model saved  (PSNR: {best_val_psnr:.2f} dB)")
        phase2_stop_counter = 0
    else:
        phase2_stop_counter += 1
        print(f"  [Phase 2 patience: {phase2_stop_counter}/{PHASE2_PATIENCE}] best={best_val_psnr:.2f} dB")

    if (global_epoch % CHECKPOINT_INTERVAL) == 0:
        torch.save({
            'epoch':                  global_epoch,
            'phase':                  'phase2',
            'model_state_dict':       model.state_dict(),
            'disc_state_dict':        disc.state_dict(),
            'optimizer_state_dict':   optimizer.state_dict(),
            'disc_opt_state_dict':    disc_optimizer.state_dict(),
            'best_val_psnr':          best_val_psnr,
            'best_gen_val_psnr':      best_gen_val_psnr,
            'phase2_lambda_adv':      phase2_lambda_adv,
            'history':                history,
        }, f'checkpoint_epoch_{global_epoch}.pth')
        print(f"  ✓ Checkpoint saved: checkpoint_epoch_{global_epoch}.pth")

    if phase2_stop_counter >= PHASE2_PATIENCE:
        print(f"\nEarly stopping Phase 2 at epoch {local_epoch + 1}/{PHASE2_EPOCHS} (patience={PHASE2_PATIENCE}).")
        break

print("\n" + "=" * 72)
print(f"Training complete.  Best Phase-1 generator PSNR: {best_gen_val_psnr:.2f} dB")
print(f"Training complete.  Best Phase-2 GAN validation PSNR: {best_val_psnr:.2f} dB")
print(f"Best generator checkpoint saved for download: {BEST_GEN_PATH}")
print(f"Phase-2 λ_adv used: {phase2_lambda_adv} (PSNR threshold: {ADV_ENABLE_PSNR_THRESH:.1f} dB)")
print(f"Positional encoding enabled at training time: {USE_POSITIONAL_ENCODING}")
print("=" * 72)


## 14. Training Visualization

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(22, 10))

labels = [
    ('train_loss',       'val_loss',       'Rec. Loss',    'Reconstruction Loss'),
    ('train_psnr',       'val_psnr',       'PSNR (dB)',    'PSNR'),
    ('train_ssim',       'val_ssim',       'SSIM',         'SSIM'),
    ('train_fft_loss',   'val_fft_loss',   'FFT L1',       'FFT Loss (high-freq)'),
    ('train_lpips_loss', 'val_lpips_loss', 'LPIPS',        'LPIPS Perceptual Loss'),
    ('train_grad_loss',  'val_grad_loss',  'Grad L1',      'Sobel Gradient Loss'),
    ('train_ssim_loss',  'val_ssim_loss',  '1 − SSIM',     'SSIM Loss Component'),
    ('train_disc_loss',  None,             'Disc. Loss',   'Discriminator Loss'),
]

for ax, (tr_key, vl_key, ylabel, title) in zip(axes.flatten(), labels):
    if history.get(tr_key):
        ax.plot(history[tr_key], label='Train', linewidth=2)
    if vl_key and history.get(vl_key):
        ax.plot(history[vl_key], label='Val', linewidth=2, linestyle='--')
    if ylabel == 'PSNR (dB)':
        ax.axhline(y=22, color='r', linestyle=':', label='Target 22 dB')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('3d_reconstruction_training_history.png', dpi=200, bbox_inches='tight')
plt.show()


## 15. Test Set Evaluation

In [ ]:
_LOCAL_CKPT = 'best_3d_reconstruction.pth'

if os.path.exists(_LOCAL_CKPT):
    _ckpt = load_checkpoint_compat(_LOCAL_CKPT, map_location=device)
    if isinstance(_ckpt, dict) and 'model_state_dict' in _ckpt:
        model.load_state_dict(_ckpt['model_state_dict'])
        if 'disc_state_dict' in _ckpt:
            disc.load_state_dict(_ckpt['disc_state_dict'])
    else:
        model.load_state_dict(_ckpt)
    print(f"Loaded weights from  {_LOCAL_CKPT}")
else:
    print("No local checkpoint found — evaluating with current (end-of-training) weights.")

model.eval()

test_metrics = validate_epoch(model, test_loader, criterion, device, scaler=None)

print("\n" + "=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"  Loss       : {test_metrics['loss']:.4f}")
print(f"  MSE        : {test_metrics['mse']:.4f}")
print(f"  SSIM Loss  : {test_metrics['ssim_loss']:.4f}")
print(f"  FFT L1     : {test_metrics['fft_loss']:.4f}")
print(f"  PSNR       : {test_metrics['psnr']:.2f} dB  (target: > 22 dB)")
print(f"  SSIM       : {test_metrics['ssim']:.4f}")
print("=" * 60)


NameError: name 'test_loader' is not defined

## 16. Visualization: Slice Reconstruction

In [ ]:
def visualize_reconstruction(model, dataloader, device, num_samples=6):
    """Show center input | target | predicted | difference for context-based pairs."""
    model.eval()
    samples = []

    with torch.no_grad():
        for batch in dataloader:
            inp = batch['input_slice'].to(device)      # (B,C,H,W), C=9 or 12
            target = batch['target_slice'].to(device)  # (B,3,H,W)
            out = model(inp)

            for i in range(min(len(inp), num_samples - len(samples))):
                samples.append({
                    'input_ctx': inp[i].cpu(),
                    'target': target[i].cpu(),
                    'output': out[i].cpu(),
                    'delta': batch['delta'][i].item(),
                })
            if len(samples) >= num_samples:
                break

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))
    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, s in enumerate(samples):
        inp_ctx_np = s['input_ctx'].numpy()      # (C,H,W), C=9 or 12
        tgt_np = s['target'].permute(1, 2, 0).numpy()
        out_np = s['output'].permute(1, 2, 0).numpy()

        # center slice FLAIR from image-context channels [z-1|z|z+1]
        center_flair = inp_ctx_np[4]

        psnr_val = psnr(tgt_np, out_np, data_range=1.0)
        ssim_val = ssim(tgt_np, out_np, data_range=1.0, channel_axis=2)

        axes[i, 0].imshow(center_flair, cmap='gray')
        axes[i, 0].set_title('Input center z (FLAIR)', fontsize=12)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(tgt_np[:, :, 1], cmap='gray')
        axes[i, 1].set_title(f'Target z+{s["delta"]}', fontsize=12)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(out_np[:, :, 1], cmap='gray')
        axes[i, 2].set_title(f'Predicted\nPSNR {psnr_val:.1f} dB', fontsize=12)
        axes[i, 2].axis('off')

        diff = np.abs(tgt_np[:, :, 1] - out_np[:, :, 1])
        im = axes[i, 3].imshow(diff, cmap='hot')
        axes[i, 3].set_title(f'Difference\nSSIM {ssim_val:.3f}', fontsize=12)
        axes[i, 3].axis('off')
        plt.colorbar(im, ax=axes[i, 3], fraction=0.046)

    plt.tight_layout()
    plt.savefig('slice_reconstruction_results.png', dpi=200, bbox_inches='tight')
    plt.show()


visualize_reconstruction(model, test_loader, device, num_samples=6)


## 17. Bidirectional 3D Volume Generation

Start from a known seed slice, generate forward autoregressively,
then run a forward-refine + backward-refine sweep to reduce drift and
accumulated directional artifacts.


In [ ]:
def _pos_map_like(slice_tensor: torch.Tensor, pos_value: float) -> torch.Tensor:
    """Create a single-channel positional map with the same H×W as a 3ch slice tensor."""
    h, w = slice_tensor.shape[-2], slice_tensor.shape[-1]
    return torch.full((1, h, w), float(np.clip(pos_value, 0.0, 1.0)), dtype=slice_tensor.dtype, device=slice_tensor.device)


def _zscore_slice_infer(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Per-slice z-score for inference tensors [3,H,W]."""
    return (x - x.mean()) / (x.std(unbiased=False) + eps)


def _build_context(prev_slice, curr_slice, next_slice, prev_pos, curr_pos, next_pos):
    """Build model input context with normalized image channels and optional positional channels."""
    prev_n = _zscore_slice_infer(prev_slice)
    curr_n = _zscore_slice_infer(curr_slice)
    next_n = _zscore_slice_infer(next_slice)

    if USE_POSITIONAL_ENCODING:
        pos_prev = _pos_map_like(curr_slice, prev_pos)
        pos_curr = _pos_map_like(curr_slice, curr_pos)
        pos_next = _pos_map_like(curr_slice, next_pos)
        return torch.cat([prev_n, curr_n, next_n, pos_prev, pos_curr, pos_next], dim=0).unsqueeze(0)
    return torch.cat([prev_n, curr_n, next_n], dim=0).unsqueeze(0)


def generate_3d_volume_bidirectional(model, seed_prev, seed_curr, num_forward=20, device='cuda',
                                     forward_refine_until_half=True):
    """Bidirectional generation for context model (positional channels optional)."""
    model.eval()
    T = num_forward + 1
    denom = max(1, T - 1)

    volume_t = torch.zeros((T, *seed_curr.shape), dtype=seed_curr.dtype, device=device)
    volume_t[0] = seed_curr.to(device)

    prev = seed_prev.to(device)
    curr = seed_curr.to(device)

    with torch.no_grad():
        # Stage A: forward generation
        for z in tqdm(range(1, T), desc='Stage A: forward generation  z→z+1'):
            prev_pos = (z - 1) / denom
            curr_pos = z / denom
            next_pos = min(z + 1, T - 1) / denom

            # next_hint unknown at inference; use current as conservative proxy
            ctx = _build_context(prev, curr, curr, prev_pos, curr_pos, next_pos)
            nxt = model(ctx)[0]
            nxt = torch.clamp(nxt, 0.0, 1.0)
            volume_t[z] = nxt
            prev, curr = curr, nxt

        # Stage B1: forward refinement
        z_stop = (T // 2) if forward_refine_until_half else T - 1
        for z in tqdm(range(1, z_stop), desc='Stage B1: forward refine'):
            p = volume_t[z - 1]
            c = volume_t[z]
            n = volume_t[z + 1] if z + 1 < T else c

            prev_pos = (z - 1) / denom
            curr_pos = z / denom
            next_pos = min(z + 1, T - 1) / denom

            ctx = _build_context(p, c, n, prev_pos, curr_pos, next_pos)
            refined = torch.clamp(model(ctx)[0], 0.0, 1.0)
            volume_t[z] = refined

        # Stage B2: backward refinement
        for z in tqdm(range(T - 2, -1, -1), desc='Stage B2: backward refine'):
            p = volume_t[z - 1] if z - 1 >= 0 else volume_t[z + 1]
            c = volume_t[z]
            n = volume_t[z + 1]

            prev_pos = max(z - 1, 0) / denom
            curr_pos = z / denom
            next_pos = (z + 1) / denom

            ctx = _build_context(p, c, n, prev_pos, curr_pos, next_pos)
            refined = torch.clamp(model(ctx)[0], 0.0, 1.0)
            volume_t[z] = refined

    return volume_t.detach().cpu().permute(0, 2, 3, 1).numpy()  # (T,H,W,3)


def visualize_3d_volume(volume, num_display=10):
    n_slices = volume.shape[0]
    indices = np.linspace(0, n_slices - 1, num_display, dtype=int)

    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    for ax, idx in zip(axes.flatten(), indices):
        ax.imshow(volume[idx, :, :, 1], cmap='gray')
        ax.set_title(f'Slice {idx}', fontsize=11)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('generated_3d_volume.png', dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:
# Pick a random test sample as seed context
seed_sample = test_dataset[np.random.randint(0, len(test_dataset))]
seed_ctx = seed_sample['input_slice']

# First 9 channels are always image context [z-1|z|z+1]
seed_prev = seed_ctx[0:3]
seed_curr = seed_ctx[3:6]

print(f"Seed context shape : {seed_ctx.shape}")
print(f"Positional encoding enabled: {USE_POSITIONAL_ENCODING}")
print("Generating 3D volume with bidirectional inference …")

volume = generate_3d_volume_bidirectional(
    model, seed_prev, seed_curr, num_forward=30, device=device,
    forward_refine_until_half=True
)

print(f"\nVolume shape : {volume.shape}  (slices, H, W, C)")
visualize_3d_volume(volume, num_display=10)


## 18. Save Model

In [ ]:
# Save final model + discriminator with training metadata
torch.save({
    'epoch':                  len(history['train_loss']),
    'model_state_dict':       model.state_dict(),
    'disc_state_dict':        disc.state_dict(),
    'optimizer_state_dict':   optimizer.state_dict(),
    'disc_opt_state_dict':    disc_optimizer.state_dict(),
    'best_val_psnr':          best_val_psnr,
    'best_generator_path':    'best_generator.pth',
    'test_psnr':              test_metrics['psnr'],
    'test_ssim':              test_metrics['ssim'],
    'history':                history,
    'config': {
        'img_size':    IMG_SIZE,
        'patch_size':  PATCH_SIZE,
        'batch_size':  BATCH_SIZE,
        'features':    (48, 96, 192),
        'max_delta':   MAX_DELTA,
        'input_channels': INPUT_CHANNELS,
        'target_channels': 3,
        'context_layout': '[z-1|z|z+1]' + (' + [pos(z-1)|pos(z)|pos(z+1)]' if USE_POSITIONAL_ENCODING else ''),
        'use_positional_encoding': USE_POSITIONAL_ENCODING,
        'rec_loss':    '2.0*MSE + 0.5*(1-SSIM) + 0.1*FFT_L1 + 0.02*LPIPS + 0.02*GradLoss',
        'adv_loss':    f'LSGAN (base lambda={LAMBDA_ADV}, phase-2 gated by Phase-1 PSNR > {ADV_ENABLE_PSNR_THRESH})',
        'ss_schedule': f'linear {SS_FACTOR_START}->{SS_FACTOR_END} over {SS_EPOCHS} epochs',
    },
}, '3d_reconstruction_final.pth')

print("Final generator + discriminator + metadata saved to  3d_reconstruction_final.pth")
print("Best generator-only checkpoint available at  best_generator.pth")


## 19. Model Summary

In [ ]:
print("\n" + "=" * 65)
print("3D VOLUME RECONSTRUCTION — SUMMARY")
print("=" * 65)
print(f"Generator        : SimpleUNet  (3 scales, features=48/96/192)")
print(f"Generator params : {total_params:,}")
disc_params = sum(p.numel() for p in disc.parameters())
print(f"Discriminator    : PatchGANDiscriminator  (70×70, ndf=64)")
print(f"Disc params      : {disc_params:,}")
context_desc = "context [z-1|z|z+1] + positional depth maps" if USE_POSITIONAL_ENCODING else "context [z-1|z|z+1] (no positional maps)"
print(f"\nTask             : {context_desc} → target z+1")
print(f"Input            : {IMG_SIZE}×{IMG_SIZE}×{INPUT_CHANNELS}")
print(f"Output           : target slice {IMG_SIZE}×{IMG_SIZE}×3")
print(f"Rec. Loss        : 2.0*MSE + 0.5*(1-SSIM) + 0.1*FFT_L1 + 0.02*LPIPS + 0.02*GradLoss")
print(f"Adv. Loss        : LSGAN base λ={LAMBDA_ADV} (enabled in Phase 2 only if Phase-1 PSNR > {ADV_ENABLE_PSNR_THRESH:.1f} dB)")
print(f"\nPatch size       : {PATCH_SIZE}×{PATCH_SIZE}")
print(f"Batch size       : {BATCH_SIZE}  (eff. {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})")
print(f"Epochs           : {NUM_EPOCHS}  (Phase1={PHASE1_EPOCHS}, Phase2={PHASE2_EPOCHS}, patience={PHASE1_PATIENCE}/{PHASE2_PATIENCE})")
print(f"\nTrain pairs      : {len(train_pairs):,}")
print(f"Val pairs        : {len(val_pairs):,}")
print(f"Test pairs       : {len(test_pairs):,}")
print(f"\nBest val PSNR    : {best_val_psnr:.2f} dB")
print(f"Test PSNR        : {test_metrics['psnr']:.2f} dB")
print(f"Test SSIM        : {test_metrics['ssim']:.4f}")
print(f"\nInference        : bidirectional generation ({'with' if USE_POSITIONAL_ENCODING else 'without'} positional refinement)")
print("=" * 65)
